# Member Retention Prediction: A Recreational Sports Club

**Important:** this notebook runs on a **fully synthetic** dataset (`data/synthetic_members.csv`). It reproduces the layout and the data problems of a real club-management export I cleaned at work, but no real member appears anywhere. Retention differs by membership group *by construction*, so the results show that the pipeline works, not anything about a real club.

## Business problem and research questions
Retention drives revenue, staffing, and scheduling. This notebook asks questions that data can answer:
1. Which member characteristics (membership group, age) are associated with being retained?
2. Can a model identify members at elevated risk of not being retained, using only the fields a club always has?
3. Does the model perform similarly across age bands (a basic fairness check)?

## How to run
From the repository root: `pip install pandas scikit-learn matplotlib seaborn`, then open this notebook and run all cells. The first code cell regenerates the data and runs the cleaning script.

In [1]:
import subprocess, sys, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = ROOT / "outputs"; OUT.mkdir(exist_ok=True)
run = lambda *a: subprocess.run([sys.executable, "-I", *map(str, a)], cwd=ROOT, check=True, capture_output=True, text=True)
run("src/generate_synthetic_members.py", "data/synthetic_members.csv")
print(run("src/clean_members.py", "data/synthetic_members.csv", "outputs/members_clean.csv").stdout)

{
  "rows_read": 7320,
  "duplicates_dropped": 151,
  "ages_set_to_missing": 31,
  "unmapped_group_labels": {},
  "rows_written": 7169,
  "columns_dropped": [
    "First name",
    "Last name",
    "Email",
    "Phone (Cell)",
    "Phone (Home)"
  ]
}



## Data explanation and preparation
The raw export has eight columns: `First name`, `Last name`, `Age`, `Email`, `Phone (Cell)`, `Phone (Home)`, `Member Status`, `User Group Name`. A report title sits above the header. `src/clean_members.py` does the following, and the log above records the counts:

| Step | Why |
|---|---|
| Find the real header row, strip whitespace, drop blank rows | Exports put a title above the headers |
| Map group codes and spellings (`SIN`, `sin`, `Single`) to one label | The same product appeared under several names |
| Drop duplicate people (same name plus email or phone) | Duplicates inflate counts and leak between train and test |
| Set impossible ages (outside 5 to 100) to missing, not guessed | Avoids inventing data; imputed later inside the model pipeline |
| Drop names, emails, and phones; keep an anonymous `member_id` | Privacy: analysis needs none of them |

**Data dictionary (clean file):** `member_id` anonymous hash; `age` years (missing allowed); `status` Active, Expired, Canceled, On Hold, or Non-member; `group` harmonized membership type; `segment` Member or Non-member.

In [2]:
df = pd.read_csv(OUT / "members_clean.csv")
print(df.shape)
display(df.groupby("segment").size().rename("rows").to_frame())
display(pd.crosstab(df["group"], df["status"]))

(7169, 5)


,rows
segment,
Member,5863
Non-member,1306


status,Active,Canceled,Expired,Non-member,On Hold
group,,,,,
Couple,346,39,317,0,51
Family,439,133,769,0,103
Guest,0,0,0,1306,0
Junior,258,43,238,0,25
Senior Couple (Monthly),90,15,92,0,21
Senior Single,237,27,183,0,33
Senior Single (Monthly),133,13,116,0,15
Single,678,183,1104,0,162


## Defining "retained" (the label)
An earlier draft counted only `Active` as retained, which wrongly treated guests (`Non-member`) as failures. Here:
- **Non-members (guests) are excluded.** They never held a membership, so they cannot be retained or lost.
- **Retained = 1** if `Active`; **0** if `Expired` or `Canceled`.
- **On Hold is excluded** from the main model because its outcome is undecided. A sensitivity check below includes it as retained.

In [3]:
mem = df[df["segment"] == "Member"].copy()
main = mem[mem["status"] != "On Hold"].copy()
main["retained"] = (main["status"] == "Active").astype(int)
print("Modeled rows:", len(main), "| retained rate:", round(main["retained"].mean(), 3))
rate = main.groupby("group")["retained"].agg(["mean", "size"]).sort_values("mean")
display(rate.round(3))
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(rate.index, rate["mean"], color="#2B6E8F")
ax.axvline(main["retained"].mean(), color="k", ls="--", lw=1, label="overall")
ax.set_xlabel("Share of members retained"); ax.set_title("Retention rate by membership group (synthetic data)")
ax.legend(); plt.tight_layout(); plt.savefig(OUT / "fig1_retention_by_group.png", dpi=150); plt.close()

Modeled rows: 5453 | retained rate: 0.4


,mean,size
group,,
Family,0.327,1341
Single,0.345,1965
Senior Couple (Monthly),0.457,197
Junior,0.479,539
Couple,0.493,702
Senior Single (Monthly),0.508,262
Senior Single,0.530,447


## Methods
- **Class imbalance** (the professor's flag): the retained share is below 50%, so accuracy alone is misleading. I use a **stratified** split, **class weights**, and judge models on **recall, precision, F1, and ROC-AUC**, with a "predict the majority class" baseline for comparison.
- **Logistic regression**: interpretable baseline; shows direction and size of each effect.
- **Decision tree** (shallow): readable rules for non-technical staff.
- **Random forest**: captures interactions between age and group; more flexible, less transparent.
- Features are **age** and **group** only, the fields a club always has. Attendance logs would likely improve results and are the recommended next data to collect.

In [4]:
X, y = main[["age", "group"]], main["retained"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=680)
pre = ColumnTransformer([("age", SimpleImputer(strategy="median", add_indicator=True), ["age"]),
                         ("grp", OneHotEncoder(handle_unknown="ignore"), ["group"])])
models = {
    "Baseline (majority class)": DummyClassifier(strategy="most_frequent"),
    "Logistic regression": LogisticRegression(class_weight="balanced", max_iter=1000),
    "Decision tree (depth 4)": DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=680),
    "Random forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=20, class_weight="balanced", random_state=680),
}
cv = StratifiedKFold(5, shuffle=True, random_state=680)
rows, fitted = [], {}
for name, m in models.items():
    p = Pipeline([("pre", pre), ("m", m)]).fit(Xtr, ytr); fitted[name] = p
    pred, proba = p.predict(Xte), p.predict_proba(Xte)[:, 1]
    rows.append({"model": name, "accuracy": accuracy_score(yte, pred), "precision": precision_score(yte, pred, zero_division=0),
                 "recall": recall_score(yte, pred), "F1": f1_score(yte, pred),
                 "ROC-AUC": roc_auc_score(yte, proba),
                 "CV ROC-AUC (5-fold)": cross_val_score(Pipeline([("pre", pre), ("m", m)]), Xtr, ytr, cv=cv, scoring="roc_auc").mean()})
results = pd.DataFrame(rows).set_index("model").round(3)
results.to_csv(OUT / "model_results.csv")
display(results)

,accuracy,precision,recall,F1,ROC-AUC,CV ROC-AUC (5-fold)
model,,,,,,
Baseline (majority class),0.600,0.000,0.000,0.000,0.500,0.500
Logistic regression,0.607,0.508,0.486,0.497,0.588,0.581
Decision tree (depth 4),0.592,0.490,0.498,0.494,0.565,0.571
Random forest,0.600,0.500,0.500,0.500,0.586,0.579


## Analysis
The table compares models on held-out data. Read it in this order: (1) does any model beat the baseline's ROC-AUC of 0.5, (2) is recall for the retained class reasonable, (3) does cross-validated AUC agree with the test AUC (a large gap would signal overfitting).

In [5]:
best = results["ROC-AUC"].drop("Baseline (majority class)").idxmax()
print("Best by test ROC-AUC:", best)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay(confusion_matrix(yte, fitted[best].predict(Xte)), display_labels=["Not retained", "Retained"]).plot(ax=axes[0], colorbar=False)
axes[0].set_title(f"Confusion matrix: {best}")
lr = fitted["Logistic regression"]
names = lr["pre"].get_feature_names_out(); coef = pd.Series(lr["m"].coef_[0], index=names).sort_values()
coef.plot.barh(ax=axes[1], color="#2B6E8F"); axes[1].set_title("Logistic regression coefficients"); axes[1].set_xlabel("Effect on log-odds of retention")
plt.tight_layout(); plt.savefig(OUT / "fig2_model_diagnostics.png", dpi=150); plt.close()
display(coef.round(3).to_frame("coefficient"))

Best by test ROC-AUC: Logistic regression


,coefficient
grp__group_Family,-0.434
grp__group_Single,-0.362
grp__group_Senior Couple (Monthly),-0.148
age__age,0.005
grp__group_Senior Single (Monthly),0.051
grp__group_Senior Single,0.152
age__missingindicator_age,0.186
grp__group_Couple,0.267
grp__group_Junior,0.388


## Fairness check by age band
A model should not work well for some groups and poorly for others. Here I compare recall and the share predicted "retained" across age bands on the test set. Age is a proxy for life stage, so large gaps would warrant caution before any use.

In [6]:
te = Xte.copy(); te["y"] = yte.values; te["pred"] = fitted[best].predict(Xte)
te["age_band"] = pd.cut(te["age"], [0, 17, 34, 54, 100], labels=["Under 18", "18-34", "35-54", "55+"])
fair = te.groupby("age_band", observed=True).apply(lambda g: pd.Series({
    "n": len(g), "actual retained": g["y"].mean(), "predicted retained": g["pred"].mean(),
    "recall": recall_score(g["y"], g["pred"]) if g["y"].sum() else np.nan})).round(3)
display(fair)
fair.to_csv(OUT / "fairness_by_age_band.csv")

,n,actual retained,predicted retained,recall
age_band,,,,
Under 18,102.0,0.441,1.000,1.000
18-34,199.0,0.367,0.101,0.096
35-54,381.0,0.325,0.147,0.202
55+,406.0,0.475,0.589,0.699


In [7]:
# Sensitivity check: treat On Hold as retained instead of excluding it
alt = mem.copy(); alt["retained"] = alt["status"].isin(["Active", "On Hold"]).astype(int)
Xa, ya = alt[["age", "group"]], alt["retained"]
Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(Xa, ya, test_size=0.2, stratify=ya, random_state=680)
p = Pipeline([("pre", pre), ("m", LogisticRegression(class_weight="balanced", max_iter=1000))]).fit(Xa_tr, ya_tr)
print("Logistic regression with On Hold counted as retained: ROC-AUC =", round(roc_auc_score(ya_te, p.predict_proba(Xa_te)[:, 1]), 3))
print("Main model (On Hold excluded): ROC-AUC =", results.loc["Logistic regression", "ROC-AUC"])

Logistic regression with On Hold counted as retained: ROC-AUC = 0.588
Main model (On Hold excluded): ROC-AUC = 0.588


## Conclusion
On this synthetic file, membership group carries the signal: Family and Single members are retained least (about 33% and 35%, against roughly 50% for most other groups), which the logistic regression coefficients confirm. Age adds almost nothing once group is known.

The models beat the "predict the majority" baseline but only weakly: the best test ROC-AUC is about 0.59 (baseline 0.50), and cross-validation agrees. With only age and group, this is **not strong enough to act on individual members**. It is useful for ranking *groups* for outreach. The best predictors a club has, such as visit frequency and tenure, are not in the export.

**The fairness check found large gaps.** In this run the model predicts nearly every member under 18 will be retained, and it catches only about 10% of retained members aged 18 to 34. Age band is tangled with membership group (all Juniors are under 18), so the model's group effect shows up as an age gap. This should be fixed or at least monitored before any use on real members.

**Answering the research questions (synthetic data):**
1. Group is the main associated characteristic; age is not.
2. A group-and-age model is a weak screen for individuals, so I would not deploy it as one.
3. Performance differs a lot across age bands, which is the strongest reason to collect richer data first.

## Assumptions
Status is accurate and current; each person appears once after de-duplication; group labels were mapped correctly; Active means retained.

## Limitations
Synthetic data; only age and group as features; no tenure, renewal history, or attendance; a single snapshot, so seasonality is invisible; On Hold excluded from the main model.

## Challenges and how I handled them
Inconsistent codes (mapped), duplicates (removed before splitting), impossible ages (set missing, imputed in the pipeline), class imbalance (stratified split, class weights, recall/F1/AUC), and privacy (identifiers dropped, synthetic data used for publication).

## Recommendations and implementation
1. Use group-level retention rates to focus outreach on the lowest-retention groups, with supportive offers (renewal reminders, family programming), never exclusion.
2. Start logging check-ins and join/renewal dates, then rerun this same pipeline on member-level behavior.
3. Review fairness by age band every time the model is retrained; keep contact data out of analysis files; restrict raw export access.

## Ethical assessment
| Concern | Likelihood | Impact | Mitigation |
|---|---|---|---|
| Privacy: exports hold names, emails, phones | Likely | Moderate | Identifiers dropped in `clean_members.py`; synthetic data published; raw files kept out of the repo |
| Bias: age and group act as proxies for life stage and income | Possible | High | Fairness table by age band; no decision rules based on demographics |
| Misuse: scores used to deprioritize members | Unlikely | High | Written policy: scores trigger supportive outreach only |
| Data validity: label choice changes results | Likely | Moderate | Non-members excluded; On Hold sensitivity check; changes logged |
| Unlabeled filtering | Possible | Moderate | Every removed row and changed value is counted in the cleaning log |

**Data changes:** duplicates removed, impossible ages set to missing, labels harmonized, identifiers dropped (all counted above). **Sourcing:** the published data is generated by `src/generate_synthetic_members.py`; the real export from work was never committed.